In [ ]:

# Criar o arquivo que contém o dashboard Streamlit
codigo_app = r'''
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(
    page_title="Uso da IA por faixa etária",
    page_icon="📊",
    layout="wide"
)

st.title("📊 Uso da Inteligência Artificial por Faixa Etária")
st.markdown(
    "Análise de utilização de IA generativa por idade, "
    "finalidade e país — dados oficiais do Eurostat."
)

@st.cache_data
def carregar_dados():
    url = (
        "https://ec.europa.eu/eurostat/api/dissemination/"
        "statistics/1.0/data/isoc_ai_iaiu?lang=en"
    )
    import requests
    import numpy as np

    resposta = requests.get(url, timeout=60)
    resposta.raise_for_status()
    dados = resposta.json()

    ids = dados["id"]
    tamanhos = dados["size"]
    dimensoes = dados["dimension"]
    valores = dados.get("value", {})

    registros = []
    itens = (
        valores.items()
        if isinstance(valores, dict)
        else enumerate(valores)
    )

    # Criar mapa de códigos e posições para cada dimensão
    mapas = {}
    for nome in ids:
        categorias = dimensoes[nome]["category"]["index"]
        if isinstance(categorias, dict):
            mapas[nome] = {
                int(posicao): codigo
                for codigo, posicao in categorias.items()
            }
        else:
            mapas[nome] = {
                int(posicao): codigo
                for posicao, codigo in enumerate(categorias)
            }

    for indice, valor in itens:
        if valor is None:
            continue

        coordenadas = np.unravel_index(int(indice), tamanhos)
        registro = {}

        for nome, posicao in zip(ids, coordenadas):
            codigo = mapas[nome][int(posicao)]
            rotulos = dimensoes[nome]["category"].get("label", {})
            registro[nome] = rotulos.get(codigo, codigo)

        registro["valor"] = float(valor)
        registros.append(registro)

    return pd.DataFrame(registros)

df = carregar_dados()

# Categorias gerais de idade, sem separar por gênero
faixas = {
    "Individuals, 16 to 24 years old": "16–24 anos",
    "Individuals, 25 to 34 years old": "25–34 anos",
    "Individuals, 35 to 44 years old": "35–44 anos",
    "Individuals, 45 to 54 years old": "45–54 anos",
    "Individuals, 55 to 64 years old": "55–64 anos",
    "Individuals, 65 to 74 years old": "65–74 anos",
}

indicadores = {
    "Use of generative AI tools: in the last 3 months": "Uso geral",
    "Use of generative AI tools: for formal education": "Educação",
    "Use of generative AI tools: for private purposes": "Uso pessoal",
    "Use of generative AI tools: for professional (work) purposes": "Trabalho",
}

unidade = "Percentage of individuals"

# Filtros
st.sidebar.header("Filtros")

anos = sorted(df["time"].dropna().astype(str).unique())
ano_padrao = "2025" if "2025" in anos else anos[-1]
ano = st.sidebar.selectbox(
    "Ano",
    anos,
    index=anos.index(ano_padrao)
)

paises_disponiveis = sorted(df["geo"].dropna().unique())
pais_padrao = (
    "European Union - 27 countries (from 2020)"
    if "European Union - 27 countries (from 2020)" in paises_disponiveis
    else paises_disponiveis[0]
)
pais = st.sidebar.selectbox(
    "País ou região",
    paises_disponiveis,
    index=paises_disponiveis.index(pais_padrao)
)

finalidade = st.sidebar.selectbox(
    "Indicador",
    list(indicadores.values())
)

indicador_selecionado = {
    nome: rotulo for nome, rotulo in indicadores.items()
    if rotulo == finalidade
}
codigo_indicador = list(indicador_selecionado.keys())[0]

base = df[
    (df["time"].astype(str) == ano)
    & (df["geo"] == pais)
    & (df["unit"] == unidade)
    & (df["indic_is"] == codigo_indicador)
    & (df["ind_type"].isin(faixas.keys()))
].copy()

base["Faixa etária"] = base["ind_type"].map(faixas)
base["valor"] = pd.to_numeric(base["valor"], errors="coerce")
base = base.dropna(subset=["valor"])

ordem = list(faixas.values())
base["Faixa etária"] = pd.Categorical(
    base["Faixa etária"], categories=ordem, ordered=True
)
base = base.sort_values("Faixa etária")

if base.empty:
    st.warning(
        "Não há observações disponíveis para os filtros selecionados. "
        "Experimente outro país, ano ou indicador."
    )
    st.stop()

# Indicadores principais
maior = base.loc[base["valor"].idxmax()]
menor = base.loc[base["valor"].idxmin()]
media = base["valor"].mean()

c1, c2, c3 = st.columns(3)
c1.metric("Maior percentual", f"{maior['valor']:.1f}%",
          help=f"Faixa: {maior['Faixa etária']}")
c2.metric("Menor percentual", f"{menor['valor']:.1f}%",
          help=f"Faixa: {menor['Faixa etária']}")
c3.metric("Média das faixas exibidas", f"{media:.1f}%")

st.caption(
    "Os indicadores representam percentuais da população de cada faixa "
    "etária, não a distribuição de todas as consultas feitas à IA."
)

# Gráfico principal
st.subheader("Uso de IA por faixa etária")

fig = px.bar(
    base,
    x="Faixa etária",
    y="valor",
    text="valor",
    color="valor",
    color_continuous_scale="Blues",
    labels={
        "Faixa etária": "Faixa etária",
        "valor": "Percentual de pessoas (%)"
    },
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(
    coloraxis_showscale=False,
    yaxis_title="Percentual de pessoas (%)",
    xaxis_title="Faixa etária",
    yaxis_range=[0, min(100, base["valor"].max() * 1.2)],
)
st.plotly_chart(fig, use_container_width=True)

# Comparação de finalidades
st.subheader("Comparação entre finalidades")

finalidades_disponiveis = df[
    (df["time"].astype(str) == ano)
    & (df["geo"] == pais)
    & (df["unit"] == unidade)
    & (df["indic_is"].isin(indicadores.keys()))
    & (df["ind_type"].isin(faixas.keys()))
].copy()

finalidades_disponiveis["Faixa etária"] = (
    finalidades_disponiveis["ind_type"].map(faixas)
)
finalidades_disponiveis["Finalidade"] = (
    finalidades_disponiveis["indic_is"].map(indicadores)
)
finalidades_disponiveis["valor"] = pd.to_numeric(
    finalidades_disponiveis["valor"], errors="coerce"
)
finalidades_disponiveis = finalidades_disponiveis.dropna(
    subset=["valor"]
)

fig2 = px.bar(
    finalidades_disponiveis,
    x="Faixa etária",
    y="valor",
    color="Finalidade",
    barmode="group",
    category_orders={"Faixa etária": ordem},
    labels={
        "Faixa etária": "Faixa etária",
        "valor": "Percentual de pessoas (%)",
    },
)
fig2.update_layout(yaxis_title="Percentual de pessoas (%)")
st.plotly_chart(fig2, use_container_width=True)

# Tabela
st.subheader("Dados utilizados")
st.dataframe(
    base[["Faixa etária", "valor"]].rename(
        columns={"valor": "Percentual (%)"}
    ).reset_index(drop=True),
    use_container_width=True,
    hide_index=True,
)

csv = base[["Faixa etária", "valor"]].to_csv(index=False).encode("utf-8")
st.download_button(
    "Baixar dados filtrados em CSV",
    data=csv,
    file_name="dados_uso_ia.csv",
    mime="text/csv",
)

st.divider()
st.subheader("Fonte e metodologia")
st.markdown(
    """
    **Fonte:** Eurostat, conjunto de dados `isoc_ai_iaiu`.

    **Portal oficial:** [Eurostat Data Browser](https://ec.europa.eu/eurostat/databrowser/view/isoc_ai_iaiu/default/table?lang=en)

    **Metodologia:** percentuais publicados sobre a utilização de ferramentas
    de IA generativa nos grupos selecionados. As categorias de idade foram
    selecionadas a partir dos rótulos oficiais da base.

    **Limitação:** os dados medem utilização de IA, não o conteúdo das
    perguntas digitadas por usuários em ferramentas específicas.
    """
)
'''

with open("app.py", "w", encoding="utf-8") as arquivo:
    arquivo.write(codigo_app)

print("Arquivo app.py criado com sucesso!")


Arquivo app.py criado com sucesso!


In [ ]:

%pip install -q streamlit plotly pandas requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 97.2 MB/s eta 0:00:00


In [ ]:

import subprocess
import time
import requests

processo = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501",
     "--server.headless", "true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

time.sleep(5)
print("Servidor iniciado. PID:", processo.pid)


Servidor iniciado. PID: 3336


In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
print("Processo ativo:", processo.poll() is None)


Processo ativo: True


In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [ ]:
print("Processo ativo:", processo.poll() is None)


Processo ativo: True


In [ ]:
import requests

try:
    resposta = requests.get("http://127.0.0.1:8501", timeout=10)
    print("Código HTTP:", resposta.status_code)
    print("Servidor respondeu corretamente.")
except Exception as erro:
    print("Erro ao acessar o servidor:", erro)

Código HTTP: 200
Servidor respondeu corretamente.
